# 02 — Analysis: figures and hypothesis tests

Everything here is computed by `scripts/make_figures.py`; this notebook calls it, so the two cannot disagree.
Inputs: `results/{main_grid,n_sweep,big_qubits,fs_gbdt}/runs.jsonl`, `results/noise.csv`, `results/hardware/*.json`
and the ATLAS CSV (Fig. 5 and the angle-spread check). Outputs: `results/figures/*.{pdf,png}`,
`results/summary_table.csv`, `results/h2_analysis.json`.

**Hypotheses.** H1–H4 are read verbatim from `docs/hypotheses.md`. They were written in the project guide on
7 Nov 2026, before any experiment was run, but were not formally pre-registered. Each is tested as its
"Test:" line says, on the PCA grid at N_train = 500 (`main_grid` + `big_qubits`, 5 seeds), and repeated on
the n = 4 and n = 8 slices with PCA replaced by boosted-tree feature selection (`fs_gbdt`).

**Significance rule.** A seed-paired difference whose mean is within one seed-to-seed sd is *not significant*
(with 5 seeds this is lenient: |mean| > 1 sd ≈ paired t 2.2, p ≈ 0.09). Correlations: 95% bootstrap CI over
configurations; a CI containing 0 is not significant. Configurations share data seeds, so these CIs are optimistic.

**Operational choices** (not part of the hypotheses): concentration = off-diagonal variance of the training
kernel; headline g = regularised g at λ = 0.01 (as in Fig. 4), other λ shown alongside; "best classical" =
max of the three classical models' test AUCs, which flatters the classical side (its size is measured in the
robustness section).

In [1]:
import json, sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'scripts'))
sys.path.insert(0, str(ROOT))
import make_figures as mf

mf.main()   # writes figures, summary_table.csv and h2_analysis.json
report = json.loads((ROOT / 'results' / 'h2_analysis.json').read_text())
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40); pd.set_option('display.max_colwidth', 120)
fmt = lambda d: f"{d['rho']:+.2f} [{d['ci_low']:+.2f}, {d['ci_high']:+.2f}]" + ('' if d['label'] == 'significant' else ' (n.s.)')
print(report['provenance'])
pd.DataFrame(report['hypotheses']).T

wrote results/summary_table.csv (125 configurations), results/h2_analysis.json and 16 figure files in results\figures
H1: HELD
    no configuration has a quantum kernel above the best classical baseline by more than the seed spread (fidelity: 58 classical better, 21 not significant, 0 quantum better; projected: 60 / 19 / 0 of 79). Against the RBF-SVM alone, 2 configuration(s) exceed it (~3.6 per kernel expected by chance with this lenient rule)
H2: FAILED
    headline (g_lam0.01): rho = -0.44 [-0.61, -0.22]; all lambdas: lambda=0 -0.50 [-0.65, -0.32], lambda=0.001 -0.63 [-0.74, -0.46], lambda=0.01 -0.44 [-0.61, -0.22], lambda=0.1 +0.34 [+0.09, +0.54]. Only lambda=0.1 is significantly positive, and it vanishes once log kernel variance is controlled for (partial rho +0.03 [-0.19, +0.25]). Significantly NEGATIVE (opposite to H2) at lambda=0, lambda=0.001, lambda=0.01
H3: HELD IN PART
    qubits: variance falls and AUC falls together in 4 of 15 (map, c) cells: ZZ map, reps 1 c=0.25, ZZ map

,statement,test
H1,The tuned classical baselines match or beat every quantum kernel on this data.,AUC with error bars across all configurations.
H2,Geometric difference correlates positively with the quantum-minus-classical AUC gap.,"Spearman correlation across configurations, with a confidence interval."
H3,"As qubits and depth grow at fixed bandwidth, concentration rises and AUC falls.",plot variance and AUC against qubit count.
H4,"Reducing bandwidth c reduces concentration and recovers performance, while pulling g back toward 1.","sweep c at fixed qubits; plot g, variance and AUC together."


## H1 — tuned classical baselines match or beat every quantum kernel

Test: AUC with error bars across all configurations.

![Fig. 1](../results/figures/fig1_auc_vs_qubits.png)

In [2]:
h1 = report['H1']
print(h1['verdict']['held'], '--', h1['verdict']['detail'])
display(pd.DataFrame({k: h1[k]['vs_best_classical'] for k in ('fidelity', 'projected')}).fillna(0).astype(int))
print('configurations where a quantum kernel exceeds the RBF-SVM alone:')
for c in h1['configs_where_a_quantum_kernel_beats_the_rbf_svm']:
    print('  ', c)
print()
print('Supplementary: quantum bandwidth chosen per seed by CV')
cols = ['model', 'feature_map', 'n_qubits', 'bandwidths_searched', 'auc', 'gap_vs_best', 'gap_vs_best_sd', 'gap_vs_best_label']
pd.DataFrame(h1['cv_selected_bandwidth'])[cols].round(4)

HELD -- no configuration has a quantum kernel above the best classical baseline by more than the seed spread (fidelity: 58 classical better, 21 not significant, 0 quantum better; projected: 60 / 19 / 0 of 79). Against the RBF-SVM alone, 2 configuration(s) exceed it (~3.6 per kernel expected by chance with this lenient rule)


,fidelity,projected
classical better,58,60
not significant,21,19


configurations where a quantum kernel exceeds the RBF-SVM alone:
   ZZ map, reps 1 n=2 c=0.1: vs RBF-SVM fidelity +0.0071 ± 0.0058, projected -0.0066 ± 0.0085; vs best classical fidelity +0.0017, projected -0.0121
   ZZ map, reps 2 n=2 c=0.05: vs RBF-SVM fidelity +0.0073 ± 0.0038, projected +0.0041 ± 0.0046; vs best classical fidelity +0.0018, projected -0.0014

Supplementary: quantum bandwidth chosen per seed by CV


,model,feature_map,n_qubits,bandwidths_searched,auc,gap_vs_best,gap_vs_best_sd,gap_vs_best_label
0,fidelity QSVM,"Z map, reps 2",2,5,0.7337,-0.0036,0.0045,not significant
1,fidelity QSVM,"Z map, reps 2",4,5,0.7895,0.0019,0.0037,not significant
2,fidelity QSVM,"Z map, reps 2",6,5,0.7976,-0.0001,0.0081,not significant
3,fidelity QSVM,"Z map, reps 2",8,5,0.8094,-0.0076,0.0113,not significant
4,fidelity QSVM,"Z map, reps 2",10,5,0.8170,-0.0051,0.0147,not significant
5,fidelity QSVM,"ZZ map, reps 1",2,5,0.7384,0.0011,0.0035,not significant
6,fidelity QSVM,"ZZ map, reps 1",4,5,0.7884,0.0008,0.0066,not significant
7,fidelity QSVM,"ZZ map, reps 1",6,5,0.7851,-0.0126,0.0061,classical better
8,fidelity QSVM,"ZZ map, reps 1",8,5,0.8020,-0.0150,0.0101,classical better
9,fidelity QSVM,"ZZ map, reps 1",10,5,0.7995,-0.0226,0.0165,classical better


## H2 — geometric difference correlates positively with the gap

Test: Spearman correlation across configurations, with a confidence interval.

![Fig. 2](../results/figures/fig2_gap_vs_g.png)

Besides the pooled test, three further checks: (a) within each feature map, (b) partial Spearman controlling
for log kernel variance, (c) Huang et al.'s label-aware ratio s_C / s_Q.

In [3]:
h2 = report['H2']
print(h2['verdict']['held'], '--', h2['verdict']['detail'])
rows = []
for lam in ['lambda=0', 'lambda=0.001', 'lambda=0.01', 'lambda=0.1']:
    r = {'lambda': lam, 'pooled': fmt(h2['pooled'][lam]), '(b) partial | log var': fmt(h2['partial_controlling_log_var'][lam])}
    for fm, d in h2['within_feature_map'][lam].items():
        r[f'(a) within {fm}'] = fmt(d)
    rows.append(r)
pd.DataFrame(rows).set_index('lambda').T

FAILED -- headline (g_lam0.01): rho = -0.44 [-0.61, -0.22]; all lambdas: lambda=0 -0.50 [-0.65, -0.32], lambda=0.001 -0.63 [-0.74, -0.46], lambda=0.01 -0.44 [-0.61, -0.22], lambda=0.1 +0.34 [+0.09, +0.54]. Only lambda=0.1 is significantly positive, and it vanishes once log kernel variance is controlled for (partial rho +0.03 [-0.19, +0.25]). Significantly NEGATIVE (opposite to H2) at lambda=0, lambda=0.001, lambda=0.01


lambda,lambda=0,lambda=0.001,lambda=0.01,lambda=0.1
pooled,"-0.50 [-0.65, -0.32]","-0.63 [-0.74, -0.46]","-0.44 [-0.61, -0.22]","+0.34 [+0.09, +0.54]"
(b) partial | log var,"-0.72 [-0.82, -0.59]","-0.77 [-0.84, -0.67]","-0.66 [-0.75, -0.54]","+0.03 [-0.19, +0.25] (n.s.)"
"(a) within Z map, reps 2","-0.62 [-0.86, -0.29]","-0.79 [-0.92, -0.57]","-0.77 [-0.90, -0.56]","-0.49 [-0.77, -0.10]"
"(a) within ZZ map, reps 1","-0.37 [-0.65, -0.01]","-0.43 [-0.69, -0.01]","-0.27 [-0.66, +0.21] (n.s.)","+0.50 [+0.07, +0.78]"
"(a) within ZZ map, reps 2","-0.25 [-0.55, +0.12] (n.s.)","-0.37 [-0.66, +0.06] (n.s.)","-0.06 [-0.45, +0.40] (n.s.)","+0.59 [+0.23, +0.81]"


In [4]:
la = h2['label_aware']
print('(c) log10(s_C / s_Q): pooled', fmt(la['pooled']), '| partial | log var', fmt(la['partial_controlling_log_var']))
for fm, d in la['within_feature_map'].items():
    print(f'    within {fm:16s}', fmt(d))
br = la['bound_regime']
print('s_classical min', round(br['s_classical_min'], 2), '| fraction s_classical < 1:', br['fraction_s_classical_below_1'])
print(br['interpretation'])

(c) log10(s_C / s_Q): pooled -0.63 [-0.75, -0.48] | partial | log var -0.61 [-0.73, -0.44]
    within Z map, reps 2    -0.51 [-0.77, -0.12]
    within ZZ map, reps 1   -0.56 [-0.81, -0.22]
    within ZZ map, reps 2   -0.64 [-0.82, -0.33]
s_classical min 4.39 | fraction s_classical < 1: 0.0
s = alpha^T K alpha / N equals Huang et al.'s s_K(N)/N, so their bound ~ sqrt(s) is vacuous for s >~ 1. s_classical is never below 1 here, and near-identity (concentrated) quantum kernels get s_quantum ~ 1 because they memorise the labels; the ratio s_C/s_Q therefore mostly flags memorising kernels, which generalise worst. The label-aware test is run outside the regime where the bound means anything.


## H3 — more qubits and depth at fixed bandwidth: concentration rises, AUC falls

Test: plot variance and AUC against qubit count.

![Fig. 3](../results/figures/fig3_concentration_vs_qubits.png)

![Fig. 5](../results/figures/fig5_kernel_heatmaps.png)

In [5]:
h3 = report['H3']
print(h3['verdict']['held'], '--', h3['verdict']['detail'])
cols = ['feature_map', 'bandwidth', 'n_values', 'log10_var_slope_per_qubit', 'log10_var_trend', 'auc_slope_per_qubit',
        'auc_trend', 'auc_at_chance_for_n_ge_4', 'H3_holds_here']
display(pd.DataFrame(h3['trend_vs_qubits'])[cols].round(4))
dep = pd.DataFrame(h3['depth_zz_reps2_vs_reps1'])
print('depth (ZZ reps 2 vs reps 1):', dep.var_label.value_counts().to_dict(), dep.auc_label.value_counts().to_dict())
print(h3['note'])

HELD IN PART -- qubits: variance falls and AUC falls together in 4 of 15 (map, c) cells: ZZ map, reps 1 c=0.25, ZZ map, reps 1 c=0.5, ZZ map, reps 1 c=1, ZZ map, reps 2 c=0.5; variance falls but AUC does not in: Z map, reps 2 c=0.5, Z map, reps 2 c=1, ZZ map, reps 1 c=0.1, ZZ map, reps 2 c=0.1, ZZ map, reps 2 c=0.25, ZZ map, reps 2 c=1 (AUC already at chance from n=4); the remaining cells show no significant variance fall. Depth (ZZ reps 2 vs reps 1, 25 (n, c) cells): reps 2 more concentrated in 12, AUC lower in 14 -- no consistent depth effect


,feature_map,bandwidth,n_values,log10_var_slope_per_qubit,log10_var_trend,auc_slope_per_qubit,auc_trend,auc_at_chance_for_n_ge_4,H3_holds_here
0,"Z map, reps 2",0.05,"[2, 4, 6, 8, 10]",0.0798,rises,0.0079,rises,False,False
1,"Z map, reps 2",0.10,"[2, 4, 6, 8, 10]",0.0703,rises,0.0102,rises,False,False
2,"Z map, reps 2",0.25,"[2, 4, 6, 8, 10]",0.0141,rises,0.0102,rises,False,False
3,"Z map, reps 2",0.50,"[2, 4, 6, 8, 10]",-0.1048,falls,0.0087,rises,False,False
4,"Z map, reps 2",1.00,"[2, 4, 6, 8, 10]",-0.2023,falls,0.0142,rises,False,False
5,"ZZ map, reps 1",0.05,"[2, 4, 6, 8, 10]",0.0438,rises,0.0073,rises,False,False
6,"ZZ map, reps 1",0.10,"[2, 4, 6, 8, 10]",-0.0593,falls,0.0042,rises,False,False
7,"ZZ map, reps 1",0.25,"[2, 4, 6, 8, 10]",-0.2746,falls,-0.0024,falls,False,True
8,"ZZ map, reps 1",0.50,"[2, 4, 6, 8, 10]",-0.4238,falls,-0.0190,falls,False,True
9,"ZZ map, reps 1",1.00,"[2, 4, 6, 8, 10]",-0.4812,falls,-0.0093,falls,True,True


depth (ZZ reps 2 vs reps 1): {'reps 2 more concentrated': 12, 'reps 2 less concentrated': 10, 'not significant': 3} {'reps 2 worse': 14, 'not significant': 10, 'reps 2 better': 1}
AUC is the raw test AUC, as H3 states; more qubits also means more PCA components (more information), which by itself raises AUC (the classical models gain 0.08 from n=2 to 10). Off-diagonal variance alone does not identify failure: nearly constant kernels (Z map, small c) also have small variance but still classify well in exact simulation


## H4 — smaller c: less concentration, recovered AUC, g pulled back toward 1

Test: sweep c at fixed qubits; plot g, variance and AUC together.

![Fig. 4](../results/figures/fig4_vs_bandwidth.png)

In [6]:
h4 = report['H4']
print(h4['verdict']['held'])
print('first half: ', h4['verdict']['first_half'])
print('second half:', h4['verdict']['second_half'])
cols = ['feature_map', 'n_qubits', 'dlog10_var_small_minus_large', 'concentration_reduced', 'auc_small', 'auc_large',
        'performance_recovered', 'g_lam0_small_c', 'g_lam0_large_c', 'g_lam0_monotone_toward_1',
        'g_lam0.01_small_c', 'g_lam0.01_large_c', 'g_lam0.01_argmax_c', 'g_lam0.01_monotone_toward_1']
pd.DataFrame(h4['cells'])[cols].round(3)

FIRST HALF HELD IN PART; SECOND HALF FAILED
first half:  smaller c recovers AUC in 15 of 15 (map, n) cells and raises kernel variance in 8 of 15; variance is NOT raised in Z map, reps 2 n=2, Z map, reps 2 n=4, Z map, reps 2 n=6, Z map, reps 2 n=8, Z map, reps 2 n=10, ZZ map, reps 1 n=2, ZZ map, reps 2 n=2: there the kernel flattens towards a constant at small c, so its variance falls even though AUC recovers
second half: verdict based on the headline regularised g (lambda=0.01): it falls steadily toward 1 in only 7 of 15 cells; at c=0.05 it is 2.1-4.7, never near 1, and for ZZ maps it peaks at intermediate c. Unregularised g (lambda=0) is closer to 1 at c=0.05 than at c=1 in 15 of 15 cells and falls steadily in 15 of 15, by orders of magnitude, but still ends at 2.8-398 at c=0.05: it moves toward 1 without getting there


,feature_map,n_qubits,dlog10_var_small_minus_large,concentration_reduced,auc_small,auc_large,performance_recovered,g_lam0_small_c,g_lam0_large_c,g_lam0_monotone_toward_1,g_lam0.01_small_c,g_lam0.01_large_c,g_lam0.01_argmax_c,g_lam0.01_monotone_toward_1
0,"Z map, reps 2",2,-2.741,False,0.730,0.595,True,2.847,96517.352,True,2.075,17.212,1.00,True
1,"Z map, reps 2",4,-2.120,False,0.781,0.670,True,4.224,29767.333,True,3.384,15.057,1.00,True
2,"Z map, reps 2",6,-1.483,False,0.782,0.683,True,5.007,3719.599,True,4.415,10.493,1.00,False
3,"Z map, reps 2",8,-0.981,False,0.792,0.707,True,5.210,795.302,True,4.716,8.478,1.00,False
4,"Z map, reps 2",10,-0.489,False,0.804,0.718,True,5.142,172.984,True,4.744,7.472,1.00,False
5,"ZZ map, reps 1",2,-0.419,False,0.737,0.606,True,98.366,338116.840,True,3.799,12.722,0.25,True
6,"ZZ map, reps 1",4,0.735,True,0.789,0.493,True,6.197,58137.087,True,2.844,8.422,0.25,True
7,"ZZ map, reps 1",6,1.790,True,0.790,0.498,True,6.566,7921.719,True,3.365,6.724,0.25,True
8,"ZZ map, reps 1",8,2.800,True,0.804,0.499,True,5.164,1784.927,True,3.157,5.420,0.25,False
9,"ZZ map, reps 1",10,3.798,True,0.802,0.509,True,4.284,276.578,True,2.894,5.091,0.25,True


## Noise and hardware

![Fig. 6](../results/figures/fig6_noise_and_hardware.png)

Details in `results/noise_summary.md`. The hardware AUCs use 24 test events and are demonstrations, not
statistically strong numbers.

## Robustness: PCA vs top-n boosted-tree features

The n = 4 and n = 8 slices of `main_grid` repeated with PCA replaced by the n features with the largest
XGBoost total-gain importance, from a fixed model fitted on the training events only (`configs/fs_gbdt.yaml`).
The classical baselines get the same n features. H1–H4 are re-tested on both slices with the same code.

![Fig. 7](../results/figures/fig7_robustness_feature_selection.png)

![Fig. 8](../results/figures/fig8_h2_feature_selection.png)

In [7]:
rob = report['robustness_feature_selection']
display(pd.DataFrame(rob['verdict_comparison']).T)
for side in ('pca', 'feature_selection'):
    print(f'--- {side}')
    for h in ('H1', 'H2', 'H3', 'H4'):
        v = rob[side][h]['verdict']
        print(f'{h}: {v["held"]}')
        for k, t in v.items():
            if k != 'held':
                print('     ', t)
print()
print('classical AUC:', json.dumps(rob['classical_auc'], indent=1))
print('median change, feature selection - PCA:', rob['median_change_fs_minus_pca'])
print('best-of-three test-AUC bias:', rob.get('best_of_three_test_auc_bias'))
print('angle spread (median IQR of training angles, radians):', rob['angle_spread'])
print('selected features (seed 0):', rob['selected_features'].get('n4_seed0'), rob['selected_features'].get('n8_seed0'))

,pca,feature_selection,changed
H1,HELD,HELD,False
H2,FAILED,FAILED,False
H3,HELD IN PART,HELD IN PART,False
H4,FIRST HALF HELD IN PART; SECOND HALF FAILED,FIRST HALF HELD IN PART; SECOND HALF FAILED,False


--- pca
H1: HELD
      no configuration has a quantum kernel above the best classical baseline by more than the seed spread (fidelity: 21 classical better, 9 not significant, 0 quantum better; projected: 21 / 9 / 0 of 30). Against the RBF-SVM alone, 0 configuration(s) exceed it (~1.3 per kernel expected by chance with this lenient rule)
H2: FAILED
      headline (g_lam0.01): rho = -0.62 [-0.81, -0.30]; all lambdas: lambda=0 -0.71 [-0.86, -0.49], lambda=0.001 -0.71 [-0.84, -0.47], lambda=0.01 -0.62 [-0.81, -0.30], lambda=0.1 +0.21 [-0.24, +0.56]. Significantly NEGATIVE (opposite to H2) at lambda=0, lambda=0.001, lambda=0.01
H3: HELD IN PART
      qubits: variance falls and AUC falls together in 1 of 15 (map, c) cells: ZZ map, reps 1 c=0.5; variance falls but AUC does not in: Z map, reps 2 c=0.5, Z map, reps 2 c=1, ZZ map, reps 1 c=0.1, ZZ map, reps 1 c=0.25, ZZ map, reps 1 c=1 (AUC already at chance from n=4), ZZ map, reps 2 c=0.1, ZZ map, reps 2 c=0.25, ZZ map, reps 2 c=0.5, ZZ map, re

## Verdicts (main PCA grid)

In [8]:
for h, v in report['verdicts'].items():
    print(f'{h}: {v["held"]}')
    for k, t in v.items():
        if k != 'held':
            print('    ' + t)
    print()

H1: HELD
    no configuration has a quantum kernel above the best classical baseline by more than the seed spread (fidelity: 58 classical better, 21 not significant, 0 quantum better; projected: 60 / 19 / 0 of 79). Against the RBF-SVM alone, 2 configuration(s) exceed it (~3.6 per kernel expected by chance with this lenient rule)

H2: FAILED
    headline (g_lam0.01): rho = -0.44 [-0.61, -0.22]; all lambdas: lambda=0 -0.50 [-0.65, -0.32], lambda=0.001 -0.63 [-0.74, -0.46], lambda=0.01 -0.44 [-0.61, -0.22], lambda=0.1 +0.34 [+0.09, +0.54]. Only lambda=0.1 is significantly positive, and it vanishes once log kernel variance is controlled for (partial rho +0.03 [-0.19, +0.25]). Significantly NEGATIVE (opposite to H2) at lambda=0, lambda=0.001, lambda=0.01

H3: HELD IN PART
    qubits: variance falls and AUC falls together in 4 of 15 (map, c) cells: ZZ map, reps 1 c=0.25, ZZ map, reps 1 c=0.5, ZZ map, reps 1 c=1, ZZ map, reps 2 c=0.5; variance falls but AUC does not in: Z map, reps 2 c=0.5, Z